# Feature Engineering on the **Titanic dataset**.

In [73]:
import pandas as pd
import numpy as np

df = pd.read_csv("data_titanic_kaggle.csv")
df.head()

,passengerId,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


| Column          | Meaning                                                              |
| --------------- | -------------------------------------------------------------------- |
| **PassengerId** | Unique ID assigned to each passenger                                 |
| **Survived**    | Survival outcome (0 = No, 1 = Yes)                                   |
| **Pclass**      | Passenger class (1 = 1st, 2 = 2nd, 3 = 3rd)                          |
| **Name**        | Full name of the passenger (often includes title)                    |
| **Sex**         | Gender of the passenger                                              |
| **Age**         | Age in years (some values missing)                                   |
| **SibSp**       | Number of siblings or spouses aboard                                 |
| **Parch**       | Number of parents or children aboard                                 |
| **Ticket**      | Ticket number                                                        |
| **Fare**        | Amount paid for the ticket                                           |
| **Cabin**       | Cabin number (many missing values)                                   |
| **Embarked**    | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |


# Initial Data Inspection

In [74]:
print(df.shape) # rows, col

(891, 12)


In [75]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerId  891 non-null    int64  
 1   survived     891 non-null    int64  
 2   pclass       891 non-null    int64  
 3   name         891 non-null    object 
 4   sex          891 non-null    object 
 5   age          714 non-null    float64
 6   sibsp        891 non-null    int64  
 7   parch        891 non-null    int64  
 8   ticket       891 non-null    object 
 9   fare         891 non-null    float64
 10  cabin        204 non-null    object 
 11  embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [76]:
# 1) How many missing values ?
missing = df.isna().sum()
print(missing)

passengerId      0
survived         0
pclass           0
name             0
sex              0
age            177
sibsp            0
parch            0
ticket           0
fare             0
cabin          687
embarked         2
dtype: int64


In [77]:
### AGE → Median (robust to outliers)
median = df["age"].median()

df["age"] = df["age"].fillna(median)

In [78]:
### EMBARK_TOWN → Mode

df["embarked"] = df["embarked"].fillna(
    df["embarked"].mode()[0]
)

In [79]:
missing = df.isna().sum()
print(missing)

passengerId      0
survived         0
pclass           0
name             0
sex              0
age              0
sibsp            0
parch            0
ticket           0
fare             0
cabin          687
embarked         0
dtype: int64


In [80]:
### NOTE: Generally we DROP columns with too many missing values: deck was missing >50% percent
## NOTE: Here we would not drop this column. We can use missing info to gain some insight

# df = df.drop(columns=["cabin"])

## 1 Feature from Domain Knowledge

### **Family Size**

**Why?**
Passengers traveling with family behaved differently during evacuation.


In [81]:
df['family_size'] = df['sibsp'] + df['parch'] + 1

#### NOTE: 
* `+1` includes the passenger
* Size = 1 → traveling alone

---

### **Is Alone (Binary Feature)**

In [82]:
df['is_alone'] = (df['family_size'] == 1).astype(int)

print(df.sample(5))

     passengerId  survived  pclass  \
187          188         1       1   
642          643         0       3   
796          797         1       1   
161          162         1       2   
662          663         0       1   

                                                  name     sex   age  sibsp  \
187      Romaine, Mr. Charles Hallace ("Mr C Rolmane")    male  45.0      0   
642                      Skoog, Miss. Margit Elizabeth  female   2.0      3   
796                        Leader, Dr. Alice (Farnham)  female  49.0      0   
161  Watt, Mrs. James (Elizabeth "Bessie" Inglis Mi...  female  40.0      0   
662                         Colley, Mr. Edward Pomeroy    male  47.0      0   

     parch      ticket     fare cabin embarked  family_size  is_alone  
187      0      111428  26.5500   NaN        S            1         1  
642      2      347088  27.9000   NaN        S            6         0  
796      0       17465  25.9292   D17        S            1         1  
161     

#### Note:
* Convert logic → numeric
* Binary features are ML-friendly


## 2 Feature from Text

### **Title from Name**

**Why?**
- Titles encode **gender, age, and social status**.


In [83]:
df['title'] = df['name'].str.extract(r',\s*([^\.]+)\.')
df['title'] = df['title'].str.strip()

# Check:
print(df['title'].value_counts())

title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64


---

### **Group Rare Titles**


In [84]:
rare_titles = [
    'Dr', 'Rev', 'Major', 'Col', 'Capt', 'Don',
    'Sir', 'Lady', 'the Countess', 'Jonkheer'
]

df.loc[df['title'].isin(rare_titles), 'title'] = 'Rare'

Why ?

> Reduce noise by grouping rare categories.

---


In [85]:
print(df.sample(5))

     passengerId  survived  pclass                             name     sex  \
157          158         0       3                  Corn, Mr. Harry    male   
586          587         0       2          Jarvis, Mr. John Denzil    male   
202          203         0       3       Johanson, Mr. Jakob Alfred    male   
315          316         1       3  Nilsson, Miss. Helmina Josefina  female   
303          304         1       2              Keane, Miss. Nora A  female   

      age  sibsp  parch           ticket     fare cabin embarked  family_size  \
157  30.0      0      0  SOTON/OQ 392090   8.0500   NaN        S            1   
586  47.0      0      0           237565  15.0000   NaN        S            1   
202  34.0      0      0          3101264   6.4958   NaN        S            1   
315  26.0      0      0           347470   7.8542   NaN        S            1   
303  28.0      0      0           226593  12.3500  E101        Q            1   

     is_alone title  
157         1   

## 3️. Feature from Numeric Transformation

### **Age Groups (Binning)**

**Why?**
- Models often learn better from age categories.


In [86]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 12, 18, 60, 100],
    labels=['Child', 'Teen', 'Adult', 'Senior'],
    include_lowest=True
)


#### NOTE:

* Continuous → categorical
* Domain-based bins

---

In [87]:
print(df.sample(5))

     passengerId  survived  pclass                        name   sex   age  \
632          633         1       1   Stahelin-Maeglin, Dr. Max  male  32.0   
725          726         0       3         Oreskovic, Mr. Luka  male  20.0   
811          812         0       3           Lester, Mr. James  male  39.0   
455          456         1       3          Jalsevac, Mr. Ivan  male  29.0   
547          548         1       2  Padro y Manent, Mr. Julian  male  28.0   

     sibsp  parch         ticket     fare cabin embarked  family_size  \
632      0      0          13214  30.5000   B50        C            1   
725      0      0         315094   8.6625   NaN        S            1   
811      0      0      A/4 48871  24.1500   NaN        S            1   
455      0      0         349240   7.8958   NaN        C            1   
547      0      0  SC/PARIS 2146  13.8625   NaN        C            1   

     is_alone title age_group  
632         1  Rare     Adult  
725         1    Mr     Adul

## 4️. Feature from Boolean Logic

### **Child Indicator**

In [88]:
df['is_child'] = (df['age'] < 12).astype(int)

### NOTE:

* Simple rule-based feature
* Often very predictive
* 0 means Not child and 1 means person is a child


In [89]:
print(df.sample(5))

     passengerId  survived  pclass                               name     sex  \
745          746         0       1       Crosby, Capt. Edward Gifford    male   
304          305         0       3  Williams, Mr. Howard Hugh "Harry"    male   
562          563         0       2         Norman, Mr. Robert Douglas    male   
362          363         0       3    Barbara, Mrs. (Catherine David)  female   
61            62         1       1                Icard, Miss. Amelie  female   

      age  sibsp  parch     ticket     fare cabin embarked  family_size  \
745  70.0      1      1  WE/P 5735  71.0000   B22        S            3   
304  28.0      0      0   A/5 2466   8.0500   NaN        S            1   
562  28.0      0      0     218629  13.5000   NaN        S            1   
362  45.0      0      1       2691  14.4542   NaN        C            2   
61   38.0      0      0     113572  80.0000   B28        S            1   

     is_alone title age_group  is_child  
745         0  Rare 

---

## 5️ Feature from Existing Categorical

### **Cabin Presence**
Instead of using raw `cabin` (too many missing), create another column


In [90]:
df['has_cabin'] = df['cabin'].notna().astype(int)

#### NOTE

> Missing itself can be information.

- 0 means that person does not have cabin. 

In [91]:
print(df.sample(5))

     passengerId  survived  pclass                            name     sex  \
735          736         0       3            Williams, Mr. Leslie    male   
547          548         1       2      Padro y Manent, Mr. Julian    male   
462          463         0       1               Gee, Mr. Arthur H    male   
160          161         0       3        Cribb, Mr. John Hatfield    male   
496          497         1       1  Eustis, Miss. Elizabeth Mussey  female   

      age  sibsp  parch         ticket     fare cabin embarked  family_size  \
735  28.5      0      0          54636  16.1000   NaN        S            1   
547  28.0      0      0  SC/PARIS 2146  13.8625   NaN        C            1   
462  47.0      0      0         111320  38.5000   E63        S            1   
160  44.0      0      1         371362  16.1000   NaN        S            2   
496  54.0      1      0          36947  78.2667   D20        C            2   

     is_alone title age_group  is_child  has_cabin  
735

---

## 6️ Feature from Fare

### **Fare per Person**


In [92]:
df['fare_per_person'] = df['fare'] / df['family_size']

#### NOTE:

* Normalizes fare
* Removes bias from group tickets

In [93]:
print(df.sample(5))

     passengerId  survived  pclass  \
319          320         1       1   
302          303         0       3   
550          551         1       1   
732          733         0       2   
472          473         1       2   

                                                  name     sex   age  sibsp  \
319  Spedden, Mrs. Frederic Oakley (Margaretta Corn...  female  40.0      1   
302                    Johnson, Mr. William Cahoone Jr    male  19.0      0   
550                        Thayer, Mr. John Borland Jr    male  17.0      0   
732                               Knight, Mr. Robert J    male  28.0      0   
472            West, Mrs. Edwy Arthur (Ada Mary Worth)  female  33.0      1   

     parch      ticket      fare cabin embarked  family_size  is_alone title  \
319      1       16966  134.5000   E34        C            3         0   Mrs   
302      0        LINE    0.0000   NaN        S            1         1    Mr   
550      2       17421  110.8833   C70        C         

---

## 7️ Feature from Embarked

###  **One-Hot Encoding**


In [94]:
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

#### Creates:

```
embarked_Q, embarked_S
```

#### NOTE:

* Convert categories → numbers
* Avoid dummy trap



In [95]:
print(df.sample(5))

     passengerId  survived  pclass                                   name  \
874          875         1       2  Abelson, Mrs. Samuel (Hannah Wizosky)   
208          209         1       3              Carr, Miss. Helen "Ellen"   
314          315         0       2                     Hart, Mr. Benjamin   
776          777         0       3                       Tobin, Mr. Roger   
645          646         1       1              Harper, Mr. Henry Sleeper   

        sex   age  sibsp  parch        ticket     fare cabin  family_size  \
874  female  28.0      1      0     P/PP 3381  24.0000   NaN            2   
208  female  16.0      0      0        367231   7.7500   NaN            1   
314    male  43.0      1      1  F.C.C. 13529  26.2500   NaN            3   
776    male  28.0      0      0        383121   7.7500   F38            1   
645    male  48.0      1      0      PC 17572  76.7292   D33            2   

     is_alone title age_group  is_child  has_cabin  fare_per_person  \
874

---

## 8 Feature Cleanup

In [96]:
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

#### NOTE:

* Explicit mapping
* More readable than auto-encoding


In [97]:
print(df.sample(5))

     passengerId  survived  pclass                                     name  \
625          626         0       1                    Sutton, Mr. Frederick   
807          808         0       3          Pettersson, Miss. Ellen Natalia   
55            56         1       1                        Woolner, Mr. Hugh   
492          493         0       1               Molson, Mr. Harry Markland   
415          416         0       3  Meek, Mrs. Thomas (Annie Louise Rowley)   

     sex   age  sibsp  parch  ticket     fare cabin  family_size  is_alone  \
625    0  61.0      0      0   36963  32.3208   D50            1         1   
807    1  18.0      0      0  347087   7.7750   NaN            1         1   
55     0  28.0      0      0   19947  35.5000   C52            1         1   
492    0  55.0      0      0  113787  30.5000   C30            1         1   
415    1  28.0      0      0  343095   8.0500   NaN            1         1   

    title age_group  is_child  has_cabin  fare_per_perso

# STOP

In [ ]:


---

#  Final Feature Engineering Summary

| Feature         | Type        | Concept Taught   |
| --------------- | ----------- | ---------------- |
| family_size     | Numeric     | Domain knowledge |
| is_alone        | Binary      | Logical features |
| title           | Categorical | Text extraction  |
| Rare title      | Categorical | Noise reduction  |
| age_group       | Categorical | Binning          |
| is_child        | Binary      | Rule-based       |
| has_cabin       | Binary      | Missing as info  |
| fare_per_person | Numeric     | Normalization    |
| embarked_*      | Numeric     | Encoding         |



In [310]:
# 1) Extract title from names

df['title'] = df['name'].str.extract(r', (\w+)\.')  # extract text between ',' and '.'
print(df['title'].value_counts())

title
Mr          518
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Jonkheer      1
Name: count, dtype: int64

In [311]:
# 2) Create rare title

rare_titles = ['Rev', 'Dr', 'Major', 'Col', 'Sir', 'Lady', 'Countess', 'Capt', 'Don', 'Jonkheer']
df['title'] = df['title'].replace(rare_titles, 'RareTitle')


In [312]:
# 3) Some anomalies in Sex can be cross-checked using Title: No anamolies
df[df['title'].isin(['Master', 'Mr']) & (df['sex'] != 'male')]


,passengerId,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,embarked,family_size,age_group,title


In [313]:
# Lets Look at surnames

df['Surname'] = df['name'].str.split(',').str[0]
print(df['Surname'].value_counts())


Surname
Andersson    9
Sage         7
Skoog        6
Panula       6
Johnson      6
            ..
Hanna        1
Lewy         1
Mineff       1
Haas         1
Dooley       1
Name: count, Length: 667, dtype: int64

In [314]:
# Lets create column family size

df['familysize'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = 1  # default
df.loc[df['familysize'] > 1, 'is_alone'] = 0

In [318]:
# 3) manually change title

df.loc[759, 'title'] = 'RareTitle'
print(df.isna().sum())

passengerId    0
survived       0
pclass         0
name           0
sex            0
age            0
sibsp          0
parch          0
ticket         0
fare           0
embarked       0
family_size    0
age_group      0
title          0
Surname        0
familysize     0
is_alone       0
dtype: int64
